In [5]:
pip install -q langchain-classic cassio

Note: you may need to restart the kernel to use updated packages.


In [1]:
# LangChain components to use
from langchain_community.vectorstores import Cassandra

# wraps and converts whole vector data into usable package
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper

# --- MODIFIED: Google Gemini Imports instead of OpenAI ---
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
# ---------------------------------------------------------

# Support for dataset retrieval with Hugging Face - can use
from datasets import load_dataset

# With CassIO, the engine powering the Astra DB integration in LangChain,
# you will also initialize the DB connection:
import cassio

/home/jxlee/Saas/PDF-Query/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from PyPDF2 import PdfReader

In [ ]:
# Astra DB Configuration
ASTRA_DB_APPLICATION_TOKEN="your_astra_token_here"
ASTRA_DB_ID="your_astra_db_id_here"

# Google AI Configuration
GOOGLE_API_KEY="your_google_api_key_here"


In [ ]:
#provide the path of pdf file/files.
pdfreader = PdfReader ('budget_speech.pdf')

In [ ]:
From typing_extensions import Concatenate

# read text from pdf
raw_text = ''
for i, page in enumerate(pdfreader. pages):
content = page.extract_text()
if content:
raw_text += content

In [ ]:
# intialize connection of DB
cassio. init (token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)

In [ ]:
# Note: Ensure you have your GOOGLE_API_KEY defined in your environment variables.
# LangChain will automatically look for `os.environ["GOOGLE_API_KEY"]` if you 
# don't explicitly pass it into the classes below.

# 1. Initialize the Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-pro", # You can also use "gemini-2.5-pro" or "gemini-3.1-pro-preview" 
    google_api_key=GOOGLE_API_KEY, 
    temperature=0.3
)

# 2. Initialize the Gemini Embeddings
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2", # Google's latest multimodal embedding model
    google_api_key=GOOGLE_API_KEY
)

In [ ]:
#Create your LangChain vector store ... backed by Astra DB!

astra_vector_store = Cassandra(
    embedding=embedding,
    table_name="mini_demo” ,
    session=None,
    keyspace=None,
)

In [ ]:
from langchain.text_splitter import CharacterTextSplitter
# We need to split the text using Character Text Split such that it should not increse token size
text_splitter = CharacterTextSplitter(

separator = "\n",

chunk_size = 800,

chunk_overlap = 200,

length_function = len,

)

texts = text_splitter.split_text(raw_text)

In [ ]:
texts[:50]

In [ ]:
# load dataset into the vector store

astra_vector_store.add_texts(texts[:50])
print(“Inserted %i headlings.” % len(texts[:50]))

astra_vector_index = VectorStoreIndexiWrapper (vectorstore=astra_vector_store)